# 20. Vision-Language Models

image-text 검색의 유사도, 후보 집합, template 평균을 계산합니다.

핵심 실습은 NumPy와 Python 표준 라이브러리만 사용합니다. API 호출, 모델 다운로드, GPU는 사용하지 않습니다. 수치는 고정된 작은 입력의 계산이며 실제 pretrained model의 성능이 아닙니다.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(27)

## Dual-encoder 비교 연산

벡터는 직접 만든 작은 입력입니다. 실제 CLIP 성능이나 학습 결과를 나타내지 않습니다.

In [2]:
def unit(z):
    return z/np.maximum(np.linalg.norm(z,axis=-1,keepdims=True),1e-12)
images=unit(np.array([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.]]))
texts=unit(np.array([[.9,.1,0.],[.1,.9,0.],[0.,.2,.8]]))
similarity=images@texts.T
print("similarity:\n",similarity)
print("top text:",similarity.argmax(axis=1))
assert np.array_equal(similarity.argmax(axis=1),[0,1,2])

similarity:
 [[0.9939 0.1104 0.    ]
 [0.1104 0.9939 0.2425]
 [0.     0.     0.9701]]
top text: [0 1 2]


## 양방향 contrastive 목적

행 방향은 image→text, 열 방향은 text→image입니다.

In [3]:
def log_softmax(x,axis):
    x=x-np.max(x,axis=axis,keepdims=True)
    return x-np.log(np.exp(x).sum(axis=axis,keepdims=True))
logits=similarity/.2
image_to_text=-np.diag(log_softmax(logits,axis=1)).mean()
text_to_image=-np.diag(log_softmax(logits,axis=0)).mean()
print("symmetric loss:",(image_to_text+text_to_image)/2)
assert image_to_text>=0 and text_to_image>=0

symmetric loss: 0.023400582713725655


## 후보 집합을 바꾸는 효과

가장 알맞은 후보가 빠져도 argmax는 남은 후보 하나를 반환합니다.

In [4]:
query=images[0]
restricted=texts[1:]
forced=int((query@restricted.T).argmax())
print("forced choice among labels 1 and 2:",forced+1)
assert forced+1!=0
threshold=.8
accepted=(query@restricted.T).max()>=threshold
print("accepted:",accepted)
assert not accepted

forced choice among labels 1 and 2: 1
accepted: False


## 과제

- 같은 class의 두 template 벡터를 정규화·평균·재정규화하여 prototype을 만드세요.
- gallery에 같은 의미의 두 정답이 있을 때 recall@2를 정의하고, image ID가 중복된 항목을 평가 전에 제거하세요.

In [5]:
# 과제 풀이 1
templates=np.array([[1.,.1,0.],[.8,.2,0.]])
prototype=unit(unit(templates).mean(axis=0,keepdims=True))
print("prototype:",prototype)
assert np.isclose(np.linalg.norm(prototype),1)
# 과제 풀이 2
gallery=[("a",.9,True),("a",.88,True),("b",.8,True),("c",.7,False)]
seen=set();dedup=[]
for item in gallery:
    if item[0] not in seen:dedup.append(item);seen.add(item[0])
top2=sorted(dedup,key=lambda x:-x[1])[:2]
recall2=sum(x[2]for x in top2)/sum(x[2]for x in dedup)
print("recall@2:",recall2)
assert recall2==1

prototype: [[0.9852 0.1715 0.    ]]
recall@2: 1.0


## 해석

유사도가 높아도 문장 안 물체 수나 관계가 정확한지는 따로 검사해야 합니다. 실제 image captioning은 이 검색 연산보다 많은 구성요소를 포함합니다. [CLIP](https://arxiv.org/abs/2103.00020), [Visual instruction tuning](https://arxiv.org/abs/2304.08485).

## Full Model Lab
The lecture HTML contains an inline numerical demo and a Model Lab panel. These notebooks run the corresponding pretrained models in a separate Python environment.
- [19_multimodal-I.ipynb](model-labs/19_multimodal-I.ipynb)
- [20_multimodal-II.ipynb](model-labs/20_multimodal-II.ipynb)
